In [ ]:
# Topic: HistoryExamples
# Execution group: full
# Workflow family: signal
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/HistoryExamples.md


In [ ]:
import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "HistoryExamples"
FAMILY = "signal"
np.random.seed(2026)
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"HistoryExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"HistoryExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"HistoryExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"HistoryExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "spikeTimes = sort(rand(1,100))*1;",
    "nst        = nspikeTrain(spikeTimes,'n1',.001);",
    "windowTimes = [.001 .002 .004];",
    "h=History(windowTimes);",
    "histn1=h.computeHistory(nst);",
    "figure; subplot(3,1,1); h.plot; ylabel('History Windows');",
    "subplot(3,1,2); histn1.plot;    ylabel('History Covariate for nst');",
    "figure; nst.plot;               ylabel('Neural Spike Train');",
    "clear nst;",
    "for i=1:1",
    "spikeTimes = sort(rand(1,100))*1;",
    "nst{i}=nspikeTrain(spikeTimes,'',.001);",
    "end",
    "spikeColl=nstColl(nst);",
    "windowTimes = [.001 .002 .01];",
    "h=History(windowTimes);",
    "histColl = h.computeHistory(spikeColl);",
    "figure; histColl.plot;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for HistoryExamples.")


In [ ]:
# HistoryExamples: fixture-backed history basis parity checks.
from pathlib import Path
import nstat
from scipy.io import loadmat
from nstat.compat.matlab import History

m = loadmat(Path(nstat.__file__).resolve().parents[2] / "tests/parity/fixtures/matlab_gold/HistoryExamples_gold.mat", squeeze_me=True)
edges = np.asarray(m["bin_edges_hist"], dtype=float).reshape(-1); spike_times = np.asarray(m["spike_times_hist"], dtype=float).reshape(-1); time_grid = np.asarray(m["time_grid_hist"], dtype=float).reshape(-1)
history = History(bin_edges_s=edges); H = history.computeHistory(spike_times, time_grid); filt = history.toFilter()
H_expected = np.asarray(m["H_expected_hist"], dtype=float); filt_expected = np.asarray(m["filter_expected_hist"], dtype=float).reshape(-1)

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
plt.sca(ax[0]); history.plot(); ax[0].set_title("History windows")
im = ax[1].imshow(H.T, aspect="auto", origin="lower", cmap="magma"); ax[1].set_title("History design matrix")
fig.colorbar(im, ax=ax[1], fraction=0.045, pad=0.04); plt.tight_layout(); plt.show()

assert H.shape == H_expected.shape
assert np.allclose(H, H_expected, atol=0.0)
assert np.allclose(filt, filt_expected, atol=0.0)
assert history.getNumBins() == int(np.asarray(m["n_bins_hist"], dtype=int).reshape(-1)[0])

CHECKPOINT_METRICS = {
    "history_bins": float(history.getNumBins()),
    "history_sum": float(np.sum(H)),
    "filter_sum": float(np.sum(filt)),
}
CHECKPOINT_LIMITS = {
    "history_bins": (1.0, 100.0),
    "history_sum": (0.0, 1.0e9),
    "filter_sum": (1.0, 1.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)
